# Evolving a Lunar Lander with differentiable Genetic Programming

## Installation
To install the required libraries run the command:

In [ ]:
# !pip install -r requirements-pinned.txt

## Imports
Imports from the standard genepro-multi library are done here. Any adjustments (e.g. different operators) should be made in the notebook. For example:

```
class SmoothOperator(Node):
  def __init__(self):
    super(SmoothOperator,self).__init__()
    self.arity = 1
    self.symb = "SmoothOperator"

  def _get_args_repr(self, args):
    return self._get_typical_repr(args,'before')

  def get_output(self, X):
    c_outs = self._get_child_outputs(X)
    return np.smoothOperation(c_outs[0])

  def get_output_pt(self, X):
    c_outs = self._get_child_outputs_pt(X)
    return torch.smoothOperation(c_outs[0])
```

In [ ]:
import gymnasium as gym

from genepro.node_impl import *
from genepro.evo import Evolution
from genepro.node_impl import Constant

import torch
import torch.optim as optim

import random
import os
import copy
from collections import namedtuple, deque

import matplotlib.pyplot as plt
from matplotlib import animation

SEED = 42

np.random.seed(SEED)
random.seed(SEED)

## Reinforcement Learning Setup
Here we first setup the Gymnasium environment. Please see https://gymnasium.farama.org/environments/box2d/lunar_lander/ for more information on the environment. 

Then a memory buffer is made. This is a buffer in which state transitions are stored. When the buffer reaches its maximum capacity old transitions are replaced by new ones.

A frame buffer is initialised used to later store animation frames of the environment.

In [ ]:
env = gym.make("LunarLander-v2", render_mode="rgb_array")

In [ ]:
Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward'))

class ReplayMemory(object):
    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Save a transition"""
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

    def __iadd__(self, other):
      self.memory += other.memory
      return self 

    def __add__(self, other):
      self.memory = self.memory + other.memory 
      return self

In [ ]:
frames = []

## Fitness Function

Here you get to be creative. The default setup evaluates 5 episodes of 300 frames. Think of what action to pick and what fitness function to use. The Multi-tree takes an input of $n \times d$ where $n$ is a batch of size 1.

In [ ]:
def fitness_function_pt(multitree, num_episodes=5, episode_duration=300, render=False, ignore_done=False):
  memory = ReplayMemory(10000)
  rewards = []

  for _ in range(num_episodes):
    # get initial state of the environment
    observation = env.reset()
    observation = observation[0]

    for _ in range(episode_duration):
      if render:
        frames.append(env.render())

      input_sample = torch.from_numpy(observation.reshape((1,-1))).float()

      action = torch.argmax(multitree.get_output_pt(input_sample))
      observation, reward, terminated, truncated, info = env.step(action.item())
      rewards.append(reward)
      output_sample = torch.from_numpy(observation.reshape((1,-1))).float()
      memory.push(input_sample, torch.tensor([[action.item()]]), output_sample, torch.tensor([reward]))
      if (terminated or truncated) and not ignore_done:
        break

  fitness = np.sum(rewards)

  return fitness, memory

In [ ]:
TRAIN_SEEDS = [SEED + i for i in range(10)]
VALIDATION_SEEDS = [10_000 + i for i in range(20)]

# LunarLander observation order: x, y, vx, vy, angle, angular velocity, left leg, right leg.
X, Y, VX, VY, ANGLE, ANGULAR_VELOCITY, LEFT_LEG, RIGHT_LEG = range(8)
NOOP, LEFT_ENGINE, MAIN_ENGINE, RIGHT_ENGINE = range(4)

FITNESS_STD_PENALTY = 0.25
INVALID_FITNESS = -1e9
PARSIMONY_WEIGHT = 0.05

CRASH_WEIGHTS = {
  X: 25.0,
  VX: 35.0,
  VY: 100.0,
  ANGLE: 35.0,
  ANGULAR_VELOCITY: 20.0,
}
CRASH_FAST_DESCENT_THRESHOLD = -0.3
CRASH_FAST_DESCENT_WEIGHT = 100.0

SAFE_FINAL_LIMITS = {
  VX: 0.35,
  VY: 0.25,
  ANGLE: 0.35,
}

NEAR_LANDING_BONUS_WEIGHT = 0.25
NEAR_LANDING_WEIGHTS = {
  X: 20.0,
  VX: 25.0,
  VY: 80.0,
  ANGLE: 50.0,
  LEFT_LEG: 15.0,
  RIGHT_LEG: 15.0,
}
BOTH_LEG_CONTACT_MAX_BONUS = 30.0

LOW_ALTITUDE_Y = 0.35
MAX_SAFE_DESCENT_VY = -0.25
LOW_ALTITUDE_DESCENT_WEIGHT = 2.0

MAIN_ENGINE_BRAKE_Y = 0.6
MAIN_ENGINE_BRAKE_VY = -0.25
MAIN_ENGINE_BRAKE_BONUS = 1.0

SOFT_TOUCHDOWN_MAX_Y = 0.4
SOFT_TOUCHDOWN_WEIGHT = 0.25
SOFT_TOUCHDOWN_WEIGHTS = {
  VY: 80.0,
  ANGLE: 40.0,
  LEFT_LEG: 25.0,
  RIGHT_LEG: 25.0,
}

TERMINAL_STATE_SCORE_WEIGHT = 0.1
TERMINAL_WEIGHTS = {
  "distance": 50.0,
  "velocity": 120.0,
  "balance": 40.0,
}

HEURISTIC_IMITATION_Y = 0.7
HEURISTIC_IMITATION_BONUS = 0.5
NO_MAIN_ENGINE_PENALTY = 50.0

In [ ]:
def _episode_seeds(base_seeds, num_episodes):
  if num_episodes <= len(base_seeds):
    return base_seeds[:num_episodes]
  return base_seeds + [base_seeds[-1] + i + 1 for i in range(num_episodes - len(base_seeds))]


def _bounded_quality(value):
  return 1.0 - min(abs(value), 1.0)


def _leg_contact_count(observation):
  return observation[LEFT_LEG] + observation[RIGHT_LEG]


def _weighted_state_quality(observation, weights):
  quality = 0.0
  for index, weight in weights.items():
    value = observation[index] if index in (LEFT_LEG, RIGHT_LEG) else _bounded_quality(observation[index])
    quality += weight * value
  return quality


def _crash_severity_penalty(observation):
  penalty = sum(weight * abs(observation[index]) for index, weight in CRASH_WEIGHTS.items())
  if observation[VY] < CRASH_FAST_DESCENT_THRESHOLD:
    penalty += CRASH_FAST_DESCENT_WEIGHT * (CRASH_FAST_DESCENT_THRESHOLD - observation[VY])
  return penalty


def _is_safe_final_state(observation):
  return all(abs(observation[index]) <= limit for index, limit in SAFE_FINAL_LIMITS.items())


def _near_landing_bonus(observation):
  if observation[VY] < -SAFE_FINAL_LIMITS[VY]:
    return 0.0
  safety_multiplier = 1.0 if _is_safe_final_state(observation) else 0.25
  return NEAR_LANDING_BONUS_WEIGHT * _weighted_state_quality(observation, NEAR_LANDING_WEIGHTS) * safety_multiplier


def _soft_touchdown_quality(observation):
  is_airborne = observation[Y] > SOFT_TOUCHDOWN_MAX_Y and _leg_contact_count(observation) == 0.0
  if is_airborne:
    return 0.0
  return SOFT_TOUCHDOWN_WEIGHT * _weighted_state_quality(observation, SOFT_TOUCHDOWN_WEIGHTS)


def _terminal_state_score(observation):
  distance_error = np.linalg.norm(observation[[X, Y]])
  velocity_error = np.linalg.norm(observation[[VX, VY]])
  balance_error = abs(observation[ANGLE]) + abs(observation[ANGULAR_VELOCITY])
  terminal_penalty = (
    TERMINAL_WEIGHTS["distance"] * distance_error +
    TERMINAL_WEIGHTS["velocity"] * velocity_error +
    TERMINAL_WEIGHTS["balance"] * balance_error
  )
  return -TERMINAL_STATE_SCORE_WEIGHT * terminal_penalty


def _tree_size(multitree):
  return sum(len(child) for child in multitree.children)


def _heuristic_action(observation):
  angle_target = np.clip(observation[X] * 0.5 + observation[VX], -0.4, 0.4)
  hover_target = 0.55 * abs(observation[X])
  angle_todo = (angle_target - observation[ANGLE]) * 0.5 - observation[ANGULAR_VELOCITY]
  hover_todo = (hover_target - observation[Y]) * 0.5 - observation[VY] * 0.5

  if observation[LEFT_LEG] or observation[RIGHT_LEG]:
    angle_todo = 0.0
    hover_todo = -observation[VY] * 0.5

  if hover_todo > abs(angle_todo) and hover_todo > 0.05:
    return MAIN_ENGINE
  if angle_todo < -0.05:
    return RIGHT_ENGINE
  if angle_todo > 0.05:
    return LEFT_ENGINE
  return NOOP


def _empty_eval_stats():
  return {
    "episode_returns": [],
    "episode_lengths": [],
    "successes": 0,
    "crashes": 0,
    "timeouts": 0,
    "invalid_outputs": 0,
    "action_counts": {action: 0 for action in range(4)},
    "main_engine_brake_steps": 0,
    "low_altitude_fast_descent_steps": 0,
    "final_abs_vys": [],
    "tree_size": 0,
    "parsimony_penalty": 0.0,
    "heuristic_match_steps": 0,
    "no_main_engine_episodes": 0,
  }


def _make_diagnostics(episode_returns=None, episode_lengths=None, successes=0, crashes=0, timeouts=0, invalid_outputs=0, action_counts=None, main_engine_brake_steps=0, low_altitude_fast_descent_steps=0, final_abs_vys=None, tree_size=0, parsimony_penalty=0.0, heuristic_match_steps=0, no_main_engine_episodes=0, stats=None):
  if stats is not None:
    episode_returns = stats["episode_returns"]
    episode_lengths = stats["episode_lengths"]
    successes = stats["successes"]
    crashes = stats["crashes"]
    timeouts = stats["timeouts"]
    invalid_outputs = stats["invalid_outputs"]
    action_counts = stats["action_counts"]
    main_engine_brake_steps = stats["main_engine_brake_steps"]
    low_altitude_fast_descent_steps = stats["low_altitude_fast_descent_steps"]
    final_abs_vys = stats["final_abs_vys"]
    tree_size = stats["tree_size"]
    parsimony_penalty = stats["parsimony_penalty"]
    heuristic_match_steps = stats["heuristic_match_steps"]
    no_main_engine_episodes = stats["no_main_engine_episodes"]

  episode_returns = episode_returns or []
  episode_lengths = episode_lengths or []
  action_counts = action_counts or {action: 0 for action in range(4)}
  final_abs_vys = final_abs_vys or []
  returns = np.array(episode_returns, dtype=float)

  return {
    "mean_return": float(np.mean(returns)) if len(returns) else INVALID_FITNESS,
    "std_return": float(np.std(returns)) if len(returns) else 0.0,
    "min_return": float(np.min(returns)) if len(returns) else INVALID_FITNESS,
    "max_return": float(np.max(returns)) if len(returns) else INVALID_FITNESS,
    "episode_returns": episode_returns,
    "episode_lengths": episode_lengths,
    "successes": successes,
    "crashes": crashes,
    "timeouts": timeouts,
    "invalid_outputs": invalid_outputs,
    "action_counts": action_counts,
    "main_engine_brake_steps": main_engine_brake_steps,
    "low_altitude_fast_descent_steps": low_altitude_fast_descent_steps,
    "mean_final_abs_vy": float(np.mean(final_abs_vys)) if len(final_abs_vys) else 0.0,
    "tree_size": tree_size,
    "parsimony_penalty": float(parsimony_penalty),
    "heuristic_match_steps": heuristic_match_steps,
    "no_main_engine_episodes": no_main_engine_episodes,
  }


def _tree_action(multitree, observation):
  input_sample = torch.from_numpy(observation.reshape((1, -1))).float()
  output = multitree.get_output_pt(input_sample)
  if not torch.isfinite(output).all():
    return None, input_sample
  return torch.argmax(output).item(), input_sample


def _apply_step_shaping(observation, action, stats):
  shaping_reward = 0.0

  if observation[Y] < LOW_ALTITUDE_Y and observation[VY] < MAX_SAFE_DESCENT_VY:
    stats["low_altitude_fast_descent_steps"] += 1
    shaping_reward -= LOW_ALTITUDE_DESCENT_WEIGHT * (MAX_SAFE_DESCENT_VY - observation[VY])

  if action == MAIN_ENGINE and observation[Y] < MAIN_ENGINE_BRAKE_Y and observation[VY] < MAIN_ENGINE_BRAKE_VY:
    stats["main_engine_brake_steps"] += 1
    shaping_reward += MAIN_ENGINE_BRAKE_BONUS * (MAIN_ENGINE_BRAKE_VY - observation[VY])

  return shaping_reward


def _finalize_episode_return(episode_return, observation, final_reward, terminated, used_main_engine, both_leg_contact_steps, stats):
  landed = terminated and final_reward >= 100 and observation[LEFT_LEG] == 1.0 and observation[RIGHT_LEG] == 1.0
  crashed = terminated and not landed

  stats["final_abs_vys"].append(abs(observation[VY]))
  stats["successes"] += int(landed)
  stats["crashes"] += int(crashed)

  if not used_main_engine:
    stats["no_main_engine_episodes"] += 1
    episode_return -= NO_MAIN_ENGINE_PENALTY

  episode_return += _near_landing_bonus(observation)
  episode_return += _soft_touchdown_quality(observation)
  episode_return += _terminal_state_score(observation)
  episode_return += min(both_leg_contact_steps, BOTH_LEG_CONTACT_MAX_BONUS)
  if crashed:
    episode_return -= _crash_severity_penalty(observation)

  return episode_return


def improved_fitness_function_pt(multitree, num_episodes=10, episode_duration=500, render=False, ignore_done=False):
  memory = ReplayMemory(10000)
  stats = _empty_eval_stats()
  stats["tree_size"] = _tree_size(multitree)
  stats["parsimony_penalty"] = PARSIMONY_WEIGHT * stats["tree_size"]

  for seed in _episode_seeds(TRAIN_SEEDS, num_episodes):
    observation = env.reset(seed=seed)[0]
    episode_return = 0.0
    final_reward = 0.0
    terminated = False
    truncated = False
    step_count = 0
    both_leg_contact_steps = 0
    used_main_engine = False

    for step_count in range(1, episode_duration + 1):
      if render:
        frames.append(env.render())

      action, input_sample = _tree_action(multitree, observation)
      if action is None:
        stats["invalid_outputs"] += 1
        fitness_function_pt.last_diagnostics = _make_diagnostics(stats=stats)
        return INVALID_FITNESS, memory

      heuristic_action = _heuristic_action(observation)
      heuristic_imitation_active = observation[Y] < HEURISTIC_IMITATION_Y
      stats["action_counts"][action] += 1
      used_main_engine = used_main_engine or action == MAIN_ENGINE

      next_observation, reward, terminated, truncated, info = env.step(action)
      final_reward = reward
      episode_return += reward
      memory.push(
        input_sample,
        torch.tensor([[action]]),
        torch.from_numpy(next_observation.reshape((1, -1))).float(),
        torch.tensor([reward]),
      )

      observation = next_observation
      episode_return += _apply_step_shaping(observation, action, stats)

      if heuristic_imitation_active and action == heuristic_action:
        stats["heuristic_match_steps"] += 1
        episode_return += HEURISTIC_IMITATION_BONUS

      if _leg_contact_count(observation) == 2.0 and _is_safe_final_state(observation):
        both_leg_contact_steps += 1

      if (terminated or truncated) and not ignore_done:
        break

    episode_return = _finalize_episode_return(
      episode_return,
      observation,
      final_reward,
      terminated,
      used_main_engine,
      both_leg_contact_steps,
      stats,
    )
    stats["episode_returns"].append(episode_return)
    stats["episode_lengths"].append(step_count)
    stats["timeouts"] += int(truncated or (not terminated and step_count >= episode_duration))

  fitness_function_pt.last_diagnostics = _make_diagnostics(stats=stats)
  fitness = (
    fitness_function_pt.last_diagnostics["mean_return"] -
    FITNESS_STD_PENALTY * fitness_function_pt.last_diagnostics["std_return"] -
    stats["parsimony_penalty"]
  )
  return fitness, memory

## Evolution Setup
Here the leaf and internal nodes are defined. Think about the odds of sampling a constant in this default configurations. Also think about any operators that could be useful and add them here. 

Adjust the population size (multiple of 8 if you want to use the standard tournament selection), max generations and max tree size to taste. Be aware that each of these settings can increase the runtime.

In [ ]:
num_features = env.observation_space.shape[0]
leaf_nodes = [Feature(i) for i in range(num_features)]
# leaf_nodes = leaf_nodes + [Constant()] # Think about the probability of sampling a coefficient
leaf_nodes = leaf_nodes + [Constant(), Constant(), Constant()] # Think about the probability of sampling a coefficient
# internal_nodes = [Plus(), Minus(), Times(), Sqrt(), Log(), Max(), IfGt()] #Add your own operators here
# internal_nodes = [Plus(),Minus(),Times(),Div(),Square(),Sqrt(),Log(),Sin(),Cos(),Max(),Min()] #Add your own operators here
internal_nodes = [Plus(),Minus(),Times(),Div(),Square(),Sqrt(),Log(),Sin(),Cos(),Max(),Min(),IfGt()] #Add your own operators here

evo = Evolution(
  # fitness_function_pt,
  improved_fitness_function_pt,
  internal_nodes,
  leaf_nodes,
  4,
  pop_size=64,
  # pop_size=128,
  max_gens=50,
  max_tree_size=31,
  # max_tree_size=63,
  n_jobs=8,
  verbose=True,
  init_method="ramped_half_and_half",
  elite_archive_size=0,
  seed=SEED)

## Evolve
Running this cell will use all the settings above as parameters

In [ ]:
evo.evolve()

# Test

In [ ]:
def get_test_score(tree, seeds=VALIDATION_SEEDS, episode_duration=1000):
    episode_returns = []
    episode_lengths = []
    successes = 0
    crashes = 0
    timeouts = 0
    invalid_outputs = 0

    for seed in seeds:
      observation = env.reset(seed=seed)[0]
      episode_return = 0.0
      final_reward = 0.0
      terminated = False
      truncated = False
      step_count = 0

      for step_count in range(1, episode_duration + 1):
        input_sample = torch.from_numpy(observation.reshape((1,-1))).float()
        output = tree.get_output_pt(input_sample)
        if not torch.isfinite(output).all():
          invalid_outputs += 1
          episode_return = INVALID_FITNESS
          break

        action = torch.argmax(output)
        observation, reward, terminated, truncated, info = env.step(action.item())
        final_reward = reward
        episode_return += reward
        if (terminated or truncated):
            break

      episode_returns.append(episode_return)
      episode_lengths.append(step_count)
      landed = terminated and final_reward >= 100 and observation[6] == 1.0 and observation[7] == 1.0
      successes += int(landed)
      crashes += int(terminated and not landed)
      timeouts += int(truncated or (not terminated and step_count >= episode_duration))

    return _make_diagnostics(
      episode_returns, episode_lengths, successes, crashes, timeouts, invalid_outputs=invalid_outputs
    )

best = evo.best_of_gens[-1]

print(best.get_readable_repr())
print(get_test_score(best))

## Make an animation
Here the best evolved individual is selected and one episode is rendered. Make sure to save your lunar landers over time to track progress and make comparisons.

In [ ]:
frames = []

# gist to save gif from https://gist.github.com/botforge/64cbb71780e6208172bbf03cd9293553
def save_frames_as_gif(frames, path='./', filename='evolved_lander.gif'):
  plt.figure(figsize=(frames[0].shape[1] / 72.0, frames[0].shape[0] / 72.0), dpi=72)
  patch = plt.imshow(frames[0])
  plt.axis('off')
  def animate(i):
      patch.set_data(frames[i])
  anim = animation.FuncAnimation(plt.gcf(), animate, frames = len(frames), interval=50)
  anim.save(path + filename, writer='imagemagick', fps=60)

frames = []
fitness_function_pt(best, num_episodes=1, episode_duration=500, render=True, ignore_done=False)
env.close()
save_frames_as_gif(frames)

## Play animation

<img src="evolved_lander.gif" width="750">

## Optimisation
The coefficients in the multi-tree aren't optimised. Here Q-learning (taken from https://pytorch.org/tutorials/intermediate/reinforcement_q_learning.html) is used to optimise the weights further. Incorporate coefficient optimisation in training your agent(s). Coefficient Optimisation can be expensive. Think about how often you want to optimise, when, which individuals etc.

In [ ]:
batch_size = 128
GAMMA = 0.99

constants = best.get_subtrees_consts()

if len(constants)>0:
  optimizer = optim.AdamW(constants, lr=1e-3, amsgrad=True)

for _ in range(500):

  if len(constants)>0 and len(evo.memory)>batch_size:
    target_tree = copy.deepcopy(best)

    transitions = evo.memory.sample(batch_size)
    batch = Transition(*zip(*transitions))
    
    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                        batch.next_state)), dtype=torch.bool)

    non_final_next_states = torch.cat([s for s in batch.next_state
                                               if s is not None])
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    state_action_values = best.get_output_pt(state_batch).gather(1, action_batch)
    next_state_values = torch.zeros(batch_size, dtype=torch.float)
    with torch.no_grad():
      next_state_values[non_final_mask] = target_tree.get_output_pt(non_final_next_states).max(1)[0].float()

    expected_state_action_values = (next_state_values * GAMMA) + reward_batch
    
    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))
   
    # Optimize the model
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_value_(constants, 100)
    optimizer.step()

print(best.get_readable_repr())
print(get_test_score(best))

In [ ]:
frames = []
fitness_function_pt(best, num_episodes=1, episode_duration=500, render=True, ignore_done=False)
env.close()
save_frames_as_gif(frames, filename='evolved_lander_RL.gif')

<img src="evolved_lander_RL.gif" width="750">